# BD6 — Stroke Rehabilitation Movement Classification Pipeline

This notebook implements the full ML pipeline for classifying 4 upper-limb rehabilitation movements:
1. **Reach & Retrieve**
2. **Cup to Lip**
3. **Arm Swing** (horizontal)
4. **Wrist Rotation**

**Pipeline stages:**
1. Data Loading & Labelling
2. Preprocessing (Butterworth bandpass filter)
3. Sliding Window Segmentation
4. Data Augmentation
5. Feature Extraction
6. Feature Selection (Sequential Forward Selection)
7. Classification (LDA + SVM)
8. Evaluation & Visualisation

## 1. Imports & Configuration

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import signal
from scipy.stats import kurtosis, skew, entropy
from scipy.signal import find_peaks

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = os.path.expanduser('~/BD6')

# ── Sensor ID → Body part mappings ────────────────────────────────────────
# Max & Yusuf — all movements
SENSORS_MAX_YUSUF = {
    '00B44876': 'Hand',
    '00B44805': 'Wrist',
    '00B44856': 'Elbow',
    '00B44877': 'Shoulder',
}

# Sara & Alfaf — Reach & Retrieve
SENSORS_SARA_ALFAF_RR = {
    '00B447F7': 'Hand',
    '00B44804': 'Wrist',
    '00B4486D': 'Elbow',
    '00B44846': 'Shoulder',
}

# Sara & Alfaf — Cup to Lip, Wrist Rotation, Arm Swing
SENSORS_SARA_ALFAF_OTHER = {
    '00B447FD': 'Hand',
    '00B447FA': 'Wrist',
    '00B447F1': 'Elbow',
    '00B44730': 'Shoulder',
}

# ── Sliding window parameters ──────────────────────────────────────────────
SAMPLING_RATE = 100          # Hz (Xsens MTW default)
WINDOW_SEC    = 2.0          # seconds per window
OVERLAP       = 0.5          # 50% overlap
WINDOW_SIZE   = int(WINDOW_SEC * SAMPLING_RATE)   # 200 samples
STEP_SIZE     = int(WINDOW_SIZE * (1 - OVERLAP))  # 100 samples

print(f'Window size : {WINDOW_SIZE} samples ({WINDOW_SEC}s)')
print(f'Step size   : {STEP_SIZE} samples')
print(f'Base dir    : {BASE_DIR}')

## 2. Data Loading

Walks through every `.txt` file, extracts the device ID from the header, maps it to a body part, and labels the recording by task and participant.

In [ ]:
def infer_task(path):
    """Return task label from file path."""
    p = path.lower()
    if any(k in p for k in ['reach and r', 'r and r', 'rr2', 'rr_new', '/rr/', 'reach and retrieve', '/reach/']):
        return 'reach_retrieve'
    if any(k in p for k in ['cup to lip', '/cup/', 'cup\\']):
        return 'cup_to_lip'
    if any(k in p for k in ['wrist rotation', '/wrist/']):
        return 'wrist_rotation'
    if any(k in p for k in ['arm swing', 'horizontal']):
        return 'arm_swing'
    return None

def infer_participant(path):
    """Return participant name from file path."""
    p = path
    if 'yussuf' in p.lower() or 'yusuf' in p.lower():
        return 'Yusuf'
    if 'max' in p.lower():
        return 'Max'
    if 'sara' in p.lower():
        return 'Sara'
    if 'alfaf' in p.lower():
        return 'Alfaf'
    return 'Unknown'

def get_sensor_map(participant, task):
    """Return the correct sensor ID→body part dict."""
    if participant in ('Max', 'Yusuf'):
        return SENSORS_MAX_YUSUF
    if task == 'reach_retrieve':
        return SENSORS_SARA_ALFAF_RR
    return SENSORS_SARA_ALFAF_OTHER

def read_mtx_file(filepath):
    """Parse one Xsens .txt export. Returns (device_id, df)."""
    device_id = None

    with open(filepath, 'r') as f:
        for line in f:
            if 'DeviceId' in line:
                device_id = line.strip().split(':')[-1].strip()
                break

    try:
        # Header is always at row 12 (0-indexed); data starts row 13
        # SampleTimeFine is always empty (double-tab artifact) so we only keep Roll/Pitch/Yaw
        df = pd.read_csv(filepath, sep='\t', skiprows=12, engine='python')
        df.columns = df.columns.str.strip()
        df = df[['Roll', 'Pitch', 'Yaw']].dropna()
        df = df.astype(float)
    except Exception:
        return device_id, None

    return device_id, df


# ── Walk all .txt files ────────────────────────────────────────────────────
records = []

for root, dirs, files in os.walk(BASE_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for fname in files:
        if not fname.endswith('.txt'):
            continue
        fpath = os.path.join(root, fname)
        task = infer_task(fpath)
        if task is None:
            continue
        participant = infer_participant(fpath)
        device_id, df = read_mtx_file(fpath)
        if df is None or len(df) < WINDOW_SIZE:
            continue

        sensor_map = get_sensor_map(participant, task)
        body_part  = sensor_map.get(device_id, 'Unknown') if device_id else 'Unknown'

        records.append({
            'participant': participant,
            'task':        task,
            'body_part':   body_part,
            'device_id':   device_id,
            'path':        fpath,
            'data':        df,
            'n_samples':   len(df),
        })

print(f'Loaded {len(records)} sensor recordings')

summary = pd.DataFrame([{k: v for k, v in r.items() if k != 'data'} for r in records])
print('\nRecordings per task:')
print(summary.groupby('task')['n_samples'].agg(['count','sum','mean'])
      .rename(columns={'count':'files','sum':'total_samples','mean':'avg_samples'}).round(0))
print('\nRecordings per participant x task:')
print(summary.groupby(['participant','task'])['body_part'].count().unstack(fill_value=0))

## 3. Preprocessing — Butterworth Bandpass Filter

As specified in the brief:
- **High-pass:** 0.1 Hz (removes slow drift/gravity components)
- **Low-pass:** 12 Hz (removes high-frequency noise)

In [ ]:
def butterworth_bandpass(data, fs=100, lowcut=0.1, highcut=12.0, order=3):
    """
    Apply a 3rd-order Butterworth bandpass filter.
    data : numpy array shape (n_samples, 3)  — Roll, Pitch, Yaw
    """
    nyq  = 0.5 * fs
    low  = lowcut  / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data, axis=0)


# Apply filter to every recording
for r in records:
    raw = r['data'].values   # already (n, 3) — Roll, Pitch, Yaw
    r['filtered'] = butterworth_bandpass(raw)

# ── Visualise raw vs filtered for one example ──────────────────────────────
ex = records[0]
t  = np.arange(len(ex['data'])) / SAMPLING_RATE
SIGNAL_COLS = ['Roll', 'Pitch', 'Yaw']

fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)
fig.suptitle(f"Raw vs Filtered — {ex['participant']} | {ex['task']} | {ex['body_part']}", fontsize=13)

for i, col in enumerate(SIGNAL_COLS):
    axes[i, 0].plot(t, ex['data'][col].values, linewidth=0.6, color='steelblue')
    axes[i, 0].set_ylabel(col)
    axes[i, 0].set_title('Raw' if i == 0 else '')

    axes[i, 1].plot(t, ex['filtered'][:, i], linewidth=0.6, color='darkorange')
    axes[i, 1].set_title('Filtered (0.1–12 Hz)' if i == 0 else '')

axes[2, 0].set_xlabel('Time (s)')
axes[2, 1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_filter.png'), dpi=120)
plt.show()
print('Filter applied to all recordings.')

## 4. Sliding Window Segmentation

Each long recording is cut into overlapping windows. For each task, we only use the **4 sensors together** (Hand, Wrist, Elbow, Shoulder) so that each window is a 200×12 matrix (4 sensors × 3 axes).

Windows where any sensor is missing are discarded.

In [ ]:
def sliding_windows(signal_array, window_size, step_size):
    """Return list of (window_size, n_cols) arrays."""
    n = len(signal_array)
    windows = []
    start = 0
    while start + window_size <= n:
        windows.append(signal_array[start:start + window_size])
        start += step_size
    return windows


# Group recordings by (participant, task, session) and combine all 4 sensors
# We identify sessions by their parent folder path
from collections import defaultdict

def session_key(r):
    return (r['participant'], r['task'], os.path.dirname(r['path']))

sessions = defaultdict(list)
for r in records:
    sessions[session_key(r)].append(r)

BODY_PARTS_ORDER = ['Hand', 'Wrist', 'Elbow', 'Shoulder']

X_windows = []   # each entry: (200, 12) array
y_labels  = []   # task label
meta      = []   # (participant, task, session_dir)

for key, recs in sessions.items():
    participant, task, session_dir = key

    # Map body_part → filtered signal
    part_map = {r['body_part']: r['filtered'] for r in recs}

    if not all(bp in part_map for bp in BODY_PARTS_ORDER):
        # Skip sessions with missing sensors
        missing = [bp for bp in BODY_PARTS_ORDER if bp not in part_map]
        print(f'  Skipping {participant}/{task} — missing: {missing}')
        continue

    # Trim all sensors to the same length
    min_len = min(len(part_map[bp]) for bp in BODY_PARTS_ORDER)
    combined = np.hstack([part_map[bp][:min_len] for bp in BODY_PARTS_ORDER])  # (n, 12)

    wins = sliding_windows(combined, WINDOW_SIZE, STEP_SIZE)
    for w in wins:
        X_windows.append(w)
        y_labels.append(task)
        meta.append((participant, task, session_dir))

X_windows = np.array(X_windows)   # (n_windows, 200, 12)
y_labels  = np.array(y_labels)

print(f'\nTotal windows : {len(X_windows)}')
print(f'Window shape  : {X_windows[0].shape}  (samples × channels)')
print(f'\nWindows per task:')
unique, counts = np.unique(y_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u:<20} : {c}')

## 5. Data Augmentation

We apply three augmentation techniques to **increase data diversity**:

| Technique | What it does | Why it's valid |
|-----------|-------------|----------------|
| **Jittering** | Adds small Gaussian noise (σ=0.5°) | Simulates sensor noise variability |
| **Scaling** | Multiplies amplitude by 0.9–1.1 | Simulates different movement magnitudes |
| **Time warping** | Stretches/compresses time axis | Simulates different movement speeds |

Each original window produces **3 augmented copies** → ~4× more data.

In [ ]:
def augment_jitter(window, sigma=0.5):
    """Add small Gaussian noise."""
    return window + np.random.normal(0, sigma, window.shape)

def augment_scale(window, scale_range=(0.9, 1.1)):
    """Multiply by a random scalar."""
    scale = np.random.uniform(*scale_range)
    return window * scale

def augment_time_warp(window, sigma=0.1):
    """
    Time warping: generate a smooth random warp curve and resample.
    Uses cubic interpolation to stretch/compress the time axis.
    """
    from scipy.interpolate import CubicSpline
    n = len(window)
    # Create random warp anchors (4 points)
    n_knots  = 4
    tt       = np.linspace(0, 1, n_knots)
    warp_pts = tt + np.random.normal(0, sigma, n_knots)
    warp_pts[0], warp_pts[-1] = 0.0, 1.0           # keep endpoints fixed
    warp_pts = np.clip(warp_pts, 0, 1)
    warp_pts = np.sort(warp_pts)

    cs      = CubicSpline(tt, warp_pts)
    t_orig  = np.linspace(0, 1, n)
    t_warp  = cs(t_orig)
    t_warp  = np.clip(t_warp, 0, 1)

    warped = np.zeros_like(window)
    for ch in range(window.shape[1]):
        warped[:, ch] = np.interp(t_orig, t_warp, window[:, ch])
    return warped


# Apply augmentation
X_aug  = []
y_aug  = []

for win, label in zip(X_windows, y_labels):
    # Keep original
    X_aug.append(win)
    y_aug.append(label)
    # Jitter
    X_aug.append(augment_jitter(win))
    y_aug.append(label)
    # Scale
    X_aug.append(augment_scale(win))
    y_aug.append(label)
    # Time warp
    X_aug.append(augment_time_warp(win))
    y_aug.append(label)

X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

print(f'Before augmentation : {len(X_windows)} windows')
print(f'After augmentation  : {len(X_aug)} windows  (×4)')
print(f'\nWindows per task (augmented):')
unique, counts = np.unique(y_aug, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u:<20} : {c}')

## 6. Feature Extraction

We extract **10 time-domain features** from each channel (Roll, Pitch, Yaw) of each sensor (Hand, Wrist, Elbow, Shoulder).

| Feature | Description |
|---------|-------------|
| Standard Deviation | Spread of signal |
| RMS | Root Mean Square — overall signal power |
| Information Entropy | Signal randomness/complexity |
| Jerk Metric | Rate of change of acceleration (smoothness) |
| Peak Count | Number of movement peaks |
| Max Peak Amplitude | Largest peak value |
| Absolute Difference | Sum of sample-to-sample changes |
| Index of Dispersion | Variance/Mean — variability measure |
| Kurtosis | Peakedness of distribution |
| Skewness | Asymmetry of distribution |

**Total features = 10 features × 3 axes × 4 sensors = 120 features**

In [ ]:
def compute_entropy(x, n_bins=20):
    """Information entropy via histogram binning."""
    hist, _ = np.histogram(x, bins=n_bins, density=True)
    hist    = hist + 1e-12   # avoid log(0)
    return entropy(hist)

def compute_jerk(x, fs=100):
    """Jerk metric: RMS of the derivative of the signal."""
    dx = np.diff(x) * fs
    return np.sqrt(np.mean(dx**2))

def extract_features(window):
    """
    window : (n_samples, 12)  — 4 sensors × 3 axes
    Returns a 1-D feature vector of length 120.
    Channel order: Hand_Roll, Hand_Pitch, Hand_Yaw,
                   Wrist_Roll, ..., Shoulder_Yaw
    """
    feats = []
    for ch in range(window.shape[1]):
        x = window[:, ch]
        peaks, _ = find_peaks(np.abs(x))
        peak_amps = np.abs(x[peaks]) if len(peaks) > 0 else np.array([0.0])

        feats.extend([
            np.std(x),                             # Standard deviation
            np.sqrt(np.mean(x**2)),                # RMS
            compute_entropy(x),                    # Information entropy
            compute_jerk(x, SAMPLING_RATE),        # Jerk metric
            len(peaks),                            # Peak count
            np.max(peak_amps),                     # Max peak amplitude
            np.sum(np.abs(np.diff(x))),            # Absolute difference
            np.var(x) / (np.mean(np.abs(x)) + 1e-12),  # Index of dispersion
            kurtosis(x),                           # Kurtosis
            skew(x),                               # Skewness
        ])
    return np.array(feats)

# Build feature names for later use
FEAT_NAMES = []
feat_labels = ['Std','RMS','Entropy','Jerk','PeakCount','MaxPeak','AbsDiff','IoD','Kurtosis','Skewness']
ch_labels   = [f'{bp}_{ax}' for bp in BODY_PARTS_ORDER for ax in ['Roll','Pitch','Yaw']]
for ch in ch_labels:
    for fl in feat_labels:
        FEAT_NAMES.append(f'{ch}_{fl}')

# Extract features from all augmented windows
print('Extracting features...')
X_feat = np.array([extract_features(w) for w in X_aug])
print(f'Feature matrix shape: {X_feat.shape}  (windows × features)')
print(f'Feature names (first 10): {FEAT_NAMES[:10]}')

## 7. Feature Selection — Sequential Forward Selection (SFS)

SFS starts with 0 features and greedily adds the feature that gives the best cross-validation score at each step. This ensures we only use features that actually improve the classifier.

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y_aug)
print('Classes:', le.classes_)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_feat)

# Run SFS with LDA as the estimator (fast, good for this)
# We select up to 30 features
print('Running Sequential Forward Selection (this may take ~1 min)...')
sfs = SequentialFeatureSelector(
    LDA(),
    n_features_to_select=30,
    direction='forward',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
sfs.fit(X_scaled, y_enc)

selected_mask   = sfs.get_support()
selected_names  = [FEAT_NAMES[i] for i in np.where(selected_mask)[0]]
X_selected      = X_scaled[:, selected_mask]

print(f'\nSelected {selected_mask.sum()} features:')
for n in selected_names:
    print(f'  {n}')

### Visualise Feature Importance (class separability)

In [ ]:
# Compute between-class variance / total variance as a simple separability score
overall_mean = X_scaled.mean(axis=0)
separability = np.zeros(X_scaled.shape[1])

for cls in np.unique(y_enc):
    mask_cls = y_enc == cls
    cls_mean = X_scaled[mask_cls].mean(axis=0)
    separability += mask_cls.sum() * (cls_mean - overall_mean)**2

separability /= len(y_enc)

# Plot top 20
top20_idx   = np.argsort(separability)[::-1][:20]
top20_names = [FEAT_NAMES[i] for i in top20_idx]
top20_vals  = separability[top20_idx]

plt.figure(figsize=(12, 5))
bars = plt.bar(range(20), top20_vals, color='steelblue', edgecolor='k', linewidth=0.5)
plt.xticks(range(20), top20_names, rotation=45, ha='right', fontsize=8)
plt.ylabel('Between-class variance')
plt.title('Top 20 Features by Class Separability')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_feature_importance.png'), dpi=120)
plt.show()

## 8. Classification

We evaluate two classifiers using **5-fold stratified cross-validation**:
- **LDA** — Linear Discriminant Analysis (fast, interpretable)
- **SVM** — Support Vector Machine with RBF kernel (often more accurate)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    'LDA': LDA(),
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
}

results = {}
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_selected, y_enc, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f'{name:<15}  Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

### Confusion Matrices

In [ ]:
from sklearn.model_selection import cross_val_predict

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = le.classes_

for ax, (name, clf) in zip(axes, classifiers.items()):
    y_pred = cross_val_predict(clf, X_selected, y_enc, cv=cv)
    cm     = confusion_matrix(y_enc, y_pred)
    disp   = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}  (acc={results[name].mean():.3f})')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_confusion_matrices.png'), dpi=120)
plt.show()

# Print detailed report for the better classifier
best_name = max(results, key=lambda k: results[k].mean())
best_clf  = classifiers[best_name]
y_pred_best = cross_val_predict(best_clf, X_selected, y_enc, cv=cv)
print(f'\nDetailed report — {best_name}:')
print(classification_report(y_enc, y_pred_best, target_names=class_names))

### Cross-Validation Score Comparison

In [ ]:
plt.figure(figsize=(7, 4))
names  = list(results.keys())
means  = [results[n].mean() for n in names]
stds   = [results[n].std()  for n in names]

bars = plt.bar(names, means, yerr=stds, capsize=8,
               color=['steelblue','darkorange'], edgecolor='k', linewidth=0.8)
plt.ylim(0, 1.05)
plt.ylabel('Accuracy')
plt.title('Classifier Comparison (5-fold CV)')
for bar, m in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, m + 0.02, f'{m:.3f}',
             ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_classifier_comparison.png'), dpi=120)
plt.show()

## 9. LDA Projection — Visualise Class Separation

LDA projects the data into a space that maximises class separability. This plot shows how well the 4 movements are separated.

In [ ]:
lda_viz = LDA(n_components=2)
X_lda   = lda_viz.fit_transform(X_selected, y_enc)

plt.figure(figsize=(8, 6))
colors = ['steelblue', 'darkorange', 'green', 'red']
for i, (cls, col) in enumerate(zip(le.classes_, colors)):
    mask = y_enc == i
    plt.scatter(X_lda[mask, 0], X_lda[mask, 1],
                label=cls, alpha=0.4, s=15, color=col)

plt.xlabel('LDA Component 1')
plt.ylabel('LDA Component 2')
plt.title('LDA Projection — Movement Classes')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'plot_lda_projection.png'), dpi=120)
plt.show()

## 10. Summary

In [ ]:
print('='*55)
print('  BD6 ML PIPELINE SUMMARY')
print('='*55)
print(f'  Participants          : Alfaf, Sara, Max, Yusuf')
print(f'  Movements classified  : {len(le.classes_)}')
print(f'  Original windows      : {len(X_windows)}')
print(f'  After augmentation    : {len(X_aug)}')
print(f'  Features extracted    : {X_feat.shape[1]}')
print(f'  Features selected     : {selected_mask.sum()} (via SFS)')
print()
for name in results:
    m, s = results[name].mean(), results[name].std()
    print(f'  {name:<15} : {m:.1%} ± {s:.1%}')
print('='*55)